In [2]:
import numpy as np

In [ ]:
class HestonModelSimulator:
    def __init__(self, params: HestonParams, config: MonteCarloConfig):
        # Validate configurations before starting simulation
        params.validate()
        config.validate()

        self.params = params
        self.config = config

In [ ]:
def simulate(self):
        """
        Simulates asset and variance paths using simple returns.
        """
        p = self.params
        c = self.config
        dt = c.maturity / c.n_steps

        S = np.zeros((c.n_steps + 1, c.n_paths))
        v = np.zeros((c.n_steps + 1, c.n_paths))

        S[0] = p.s0
        v[0] = p.v0

        np.random.seed(c.seed)

        # Cholesky components
        rho = p.rho
        sqrt_rho = np.sqrt(1.0 - rho**2)

        for t in range(1, c.n_steps + 1):
            z1 = np.random.normal(0, 1, c.n_paths)
            z2 = np.random.normal(0, 1, c.n_paths)

            # Brownian increments
            dW_s = z1 * np.sqrt(dt)
            dW_v = (rho * z1 + sqrt_rho * z2) * np.sqrt(dt)

            # Full Truncation for variance stability
            v_prev = v[t-1]
            v_plus = np.maximum(v_prev, 0)

            # Variance update (remains the same)
            v[t] = v_prev + p.kappa * (p.theta - v_plus) * dt + \
                   p.sigma * np.sqrt(v_plus) * dW_v

            # Asset update using SIMPLE RETURNS
            # dS = S * (r-q) * dt + S * sqrt(v+) * dWs
            S[t] = S[t-1] * (1 + (p.r - p.q) * dt + np.sqrt(v_plus) * dW_s)

            # Safety check: Simple returns can technically push S negative
            # if vol is very high or dt is very large.
            S[t] = np.maximum(S[t], 0)

        return S, v

In [ ]:
# Example Usage:
# h_params = default_heston_params()
# mc_config = default_mc_config()
# simulator = HestonModelSimulator(h_params, mc_config)
# S_paths, v_paths = simulator.simulate()